# Illustrative Example: OSRM Base Duration, TomTom-Based Quantile Multiplier, and Leaflet Map

Notebook ini dibuat untuk memperkuat penjelasan metodologi RKP/disertasi Ayub.

Tujuan notebook:
1. Menampilkan contoh **1 tanggal pengiriman** dan **1 pasangan origin-destination**.
2. Menunjukkan bahwa waktu tempuh kuantil dibentuk dari:

\[
T^q_{ij} = T^{OSRM}_{ij} \times M_q
\]

3. Menampilkan **p50, p85, p95**, traffic penalty, dan peta Leaflet/Folium.
4. Menyimpan output ke CSV dan HTML sehingga bisa dimasukkan ke lampiran artefak digital.

Catatan metodologis:
- OSRM digunakan sebagai **driving road-network approximation**, bukan truck-specific routing.
- TomTom-based multiplier digunakan sebagai **traffic inflation factor berbasis kuantil**, bukan live traffic control penuh.
- Peta Leaflet bersifat ilustratif untuk menjelaskan origin-destination dan estimasi rute contoh.

In [ ]:
# ============================================================
# 0. INSTALL AND IMPORT LIBRARIES
# ============================================================

!pip -q install folium pandas matplotlib

import os
import zipfile
import requests
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import folium
from IPython.display import display, HTML

## 1. Upload atau Deteksi Saved Matrix Package

Notebook ini membaca file ZIP saved matrix package, misalnya:

```text
osrm_tomtom_saved_matrices.zip
osrm_tomtom_saved_matrices (1).zip
```

File tersebut minimal berisi:
- `osrm_node_index.csv`
- `combined_osrm_distance_matrix_km_long.csv`
- `combined_osrm_duration_matrix_min_long.csv`
- `combined_travel_time_matrix_p50_long.csv`
- `combined_travel_time_matrix_p85_long.csv`
- `combined_travel_time_matrix_p95_long.csv`
- `tomtom_traffic_multiplier.csv`
- `matrix_source_audit.csv`

In [ ]:
# ============================================================
# 1. SETUP PATH
# ============================================================

ZIP_PATH = "/content/osrm_tomtom_saved_matrices.zip"
ALT_ZIP_PATH = "/content/osrm_tomtom_saved_matrices (1).zip"

EXTRACT_DIR = Path("/content/osrm_tomtom_saved_matrices_extracted")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

zip_candidates = list(Path("/content").glob("osrm_tomtom_saved_matrices*.zip"))

if Path(ZIP_PATH).exists():
    selected_zip = Path(ZIP_PATH)
elif Path(ALT_ZIP_PATH).exists():
    selected_zip = Path(ALT_ZIP_PATH)
elif zip_candidates:
    selected_zip = zip_candidates[0]
else:
    from google.colab import files
    print("Upload file saved matrix package ZIP...")
    uploaded = files.upload()
    uploaded_zip = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
    if not uploaded_zip:
        raise FileNotFoundError("Tidak ada file ZIP yang diupload.")
    selected_zip = Path("/content") / uploaded_zip[0]

print("Selected ZIP:", selected_zip)

with zipfile.ZipFile(selected_zip, "r") as z:
    z.extractall(EXTRACT_DIR)

print("Extracted to:", EXTRACT_DIR)
print("Top-level CSV files:")
for f in sorted(EXTRACT_DIR.glob("*.csv")):
    print("-", f.name)

In [ ]:
# ============================================================
# 2. LOAD REQUIRED CSV FILES
# ============================================================

def read_required_csv(filename):
    path = EXTRACT_DIR / filename
    if not path.exists():
        matches = list(EXTRACT_DIR.rglob(filename))
        if not matches:
            raise FileNotFoundError(f"File tidak ditemukan: {filename}")
        path = matches[0]
    return pd.read_csv(path)

nodes_df = read_required_csv("osrm_node_index.csv")
distance_df = read_required_csv("combined_osrm_distance_matrix_km_long.csv")
duration_df = read_required_csv("combined_osrm_duration_matrix_min_long.csv")
p50_df = read_required_csv("combined_travel_time_matrix_p50_long.csv")
p85_df = read_required_csv("combined_travel_time_matrix_p85_long.csv")
p95_df = read_required_csv("combined_travel_time_matrix_p95_long.csv")
multiplier_df = read_required_csv("tomtom_traffic_multiplier.csv")

try:
    source_audit_df = read_required_csv("matrix_source_audit.csv")
except Exception:
    source_audit_df = pd.DataFrame()

for df in [distance_df, duration_df, p50_df, p85_df, p95_df]:
    df["delivery_date"] = pd.to_datetime(df["delivery_date"]).dt.strftime("%Y-%m-%d")

print("Nodes:", nodes_df.shape)
print("Distance matrix:", distance_df.shape)
print("Duration matrix:", duration_df.shape)
print("p50 matrix:", p50_df.shape)
print("p85 matrix:", p85_df.shape)
print("p95 matrix:", p95_df.shape)
print("Multiplier:", multiplier_df.shape)

print("\nMultiplier table:")
display(multiplier_df)

if not source_audit_df.empty:
    print("\nMatrix source audit:")
    display(source_audit_df)

## 2. Dasar Kuantil dan Multiplier

Interpretasi kuantil:
- **p50** = kondisi normal atau median.
- **p85** = kondisi padat; waktu tempuh berada pada upper-tail namun belum ekstrem.
- **p95** = kondisi sangat padat atau konservatif tinggi.

Secara komputasional, notebook ini membaca nilai multiplier dari file `tomtom_traffic_multiplier.csv`. Jika file tersebut tersedia, nilai multiplier digunakan langsung dari file. Jika format file berbeda, notebook menggunakan nilai final yang sudah diaudit:

| Kuantil | Multiplier | Makna |
|---|---:|---|
| p50 | 1.0000 | kondisi normal/median |
| p85 | 1.1503 | waktu tempuh naik sekitar 15,03% dari p50 |
| p95 | 1.3071 | waktu tempuh naik sekitar 30,71% dari p50 |

In [ ]:
# ============================================================
# 3. GET MULTIPLIERS
# ============================================================

def get_multiplier(multiplier_df, quantile, default_value):
    if multiplier_df is None or multiplier_df.empty:
        return default_value
    qcol_candidates = [c for c in multiplier_df.columns if c.lower() in ["quantile", "kuantil"]]
    mcol_candidates = [c for c in multiplier_df.columns if "mult" in c.lower()]
    if not qcol_candidates or not mcol_candidates:
        return default_value
    qcol = qcol_candidates[0]
    mcol = mcol_candidates[0]
    match = multiplier_df[multiplier_df[qcol].astype(str).str.lower() == quantile.lower()]
    if match.empty:
        return default_value
    return float(match.iloc[0][mcol])

m_p50 = get_multiplier(multiplier_df, "p50", 1.0000)
m_p85 = get_multiplier(multiplier_df, "p85", 1.1503)
m_p95 = get_multiplier(multiplier_df, "p95", 1.3071)

quantile_basis = pd.DataFrame({
    "quantile": ["p50", "p85", "p95"],
    "multiplier": [m_p50, m_p85, m_p95],
    "increase_vs_p50_percent": [0, (m_p85 / m_p50 - 1) * 100, (m_p95 / m_p50 - 1) * 100],
    "operational_interpretation": [
        "Kondisi normal atau median",
        "Kondisi perjalanan padat",
        "Kondisi perjalanan sangat padat atau konservatif tinggi"
    ]
})

display(quantile_basis.round(4))

## 3. Pilih Satu Tanggal dan Satu Pasangan Origin-Destination

Silakan atur variabel berikut:
- `EXAMPLE_DATE`: tanggal contoh. Gunakan `AUTO` untuk mengambil tanggal pertama yang tersedia.
- `ORIGIN_ID`: default `DEPOT_1`.
- `DESTINATION_ID`: gunakan `AUTO` untuk mengambil customer pertama pada tanggal tersebut.

Contoh default akan mengambil tanggal pertama dan pasangan `DEPOT_1 → customer terdekat` yang tersedia pada saved matrix.

In [ ]:
# ============================================================
# 4. SELECT EXAMPLE DATE AND OD PAIR
# ============================================================

EXAMPLE_DATE = "AUTO"       # contoh: "2024-03-01" atau "AUTO"
ORIGIN_ID = "DEPOT_1"        # contoh: "DEPOT_1"
DESTINATION_ID = "AUTO"      # contoh: "2301.A006" atau "AUTO"

available_dates = sorted(distance_df["delivery_date"].unique())

if EXAMPLE_DATE == "AUTO":
    example_date = available_dates[0]
else:
    example_date = pd.to_datetime(EXAMPLE_DATE).strftime("%Y-%m-%d")
    if example_date not in available_dates:
        raise ValueError(f"Tanggal {example_date} tidak tersedia. Contoh tanggal tersedia: {available_dates[:5]}")

if DESTINATION_ID == "AUTO":
    od_candidates = distance_df[
        (distance_df["delivery_date"] == example_date) &
        (distance_df["origin_id"] == ORIGIN_ID) &
        (distance_df["destination_id"] != ORIGIN_ID) &
        (distance_df["distance_km"] > 0)
    ].copy()
    if od_candidates.empty:
        raise ValueError("Tidak ada OD candidate untuk tanggal dan origin yang dipilih.")
    selected_od = od_candidates.sort_values("distance_km").iloc[0]
    origin_id = ORIGIN_ID
    destination_id = selected_od["destination_id"]
else:
    origin_id = ORIGIN_ID
    destination_id = DESTINATION_ID

print("Example date:", example_date)
print("Origin:", origin_id)
print("Destination:", destination_id)

In [ ]:
# ============================================================
# 5. EXTRACT OD VALUES AND CALCULATE QUANTILE TRAVEL TIME
# ============================================================

def get_od_value(df, value_col, delivery_date, origin_id, destination_id):
    row = df[
        (df["delivery_date"] == delivery_date) &
        (df["origin_id"] == origin_id) &
        (df["destination_id"] == destination_id)
    ]
    if row.empty:
        raise ValueError(f"OD tidak ditemukan: {delivery_date}, {origin_id} -> {destination_id}")
    return float(row.iloc[0][value_col])

distance_km = get_od_value(distance_df, "distance_km", example_date, origin_id, destination_id)
duration_min = get_od_value(duration_df, "duration_min", example_date, origin_id, destination_id)

travel_p50_saved = get_od_value(p50_df, "travel_time_min_p50", example_date, origin_id, destination_id)
travel_p85_saved = get_od_value(p85_df, "travel_time_min_p85", example_date, origin_id, destination_id)
travel_p95_saved = get_od_value(p95_df, "travel_time_min_p95", example_date, origin_id, destination_id)

travel_p50_calc = duration_min * m_p50
travel_p85_calc = duration_min * m_p85
travel_p95_calc = duration_min * m_p95

penalty_p85 = travel_p85_calc - travel_p50_calc
penalty_p95 = travel_p95_calc - travel_p50_calc

example_result = pd.DataFrame({
    "delivery_date": [example_date],
    "origin_id": [origin_id],
    "destination_id": [destination_id],
    "distance_km_osrm": [distance_km],
    "duration_min_osrm_base": [duration_min],
    "multiplier_p50": [m_p50],
    "multiplier_p85": [m_p85],
    "multiplier_p95": [m_p95],
    "travel_time_p50_calc": [travel_p50_calc],
    "travel_time_p85_calc": [travel_p85_calc],
    "travel_time_p95_calc": [travel_p95_calc],
    "traffic_penalty_p85": [penalty_p85],
    "traffic_penalty_p95": [penalty_p95],
    "travel_time_p50_saved": [travel_p50_saved],
    "travel_time_p85_saved": [travel_p85_saved],
    "travel_time_p95_saved": [travel_p95_saved],
    "delta_p50_calc_vs_saved": [travel_p50_calc - travel_p50_saved],
    "delta_p85_calc_vs_saved": [travel_p85_calc - travel_p85_saved],
    "delta_p95_calc_vs_saved": [travel_p95_calc - travel_p95_saved],
})

display(example_result.round(4))

example_csv_path = "/content/example_osrm_quantile_travel_time.csv"
example_result.to_csv(example_csv_path, index=False)
print("Saved CSV:", example_csv_path)

## 4. Visualisasi Waktu Tempuh Kuantil

Grafik berikut menunjukkan bagaimana durasi dasar OSRM diubah menjadi waktu tempuh p50, p85, dan p95 menggunakan multiplier.

In [ ]:
# ============================================================
# 6. BAR CHART OF QUANTILE TRAVEL TIME
# ============================================================

plot_df = pd.DataFrame({
    "Quantile": ["p50", "p85", "p95"],
    "Travel Time (minutes)": [travel_p50_calc, travel_p85_calc, travel_p95_calc]
})

ax = plot_df.plot(
    kind="bar",
    x="Quantile",
    y="Travel Time (minutes)",
    legend=False,
    figsize=(7, 4)
)
ax.set_title(f"Quantile Travel Time Example\n{example_date}: {origin_id} → {destination_id}")
ax.set_xlabel("Travel-time quantile")
ax.set_ylabel("Minutes")

for i, v in enumerate(plot_df["Travel Time (minutes)"]):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

## 5. Peta Leaflet/Folium

Peta berikut menampilkan depot dan customer contoh. Notebook akan mencoba memanggil **OSRM Route API** untuk menampilkan geometri rute jalan. Jika gagal, notebook tetap menampilkan garis lurus sebagai ilustrasi OD.

Catatan: geometri rute pada peta hanya untuk ilustrasi satu OD. Eksperimen utama tetap menggunakan saved matrix package agar hasil reprodusibel.

In [ ]:
# ============================================================
# 7. CREATE LEAFLET MAP WITH FOLIUM
# ============================================================

required_node_cols = {"node_id", "longitude", "latitude", "node_type"}
if not required_node_cols.issubset(set(nodes_df.columns)):
    raise ValueError(f"Kolom node tidak lengkap. Wajib ada: {required_node_cols}")

origin_node = nodes_df[nodes_df["node_id"] == origin_id]
dest_node = nodes_df[nodes_df["node_id"] == destination_id]

if origin_node.empty:
    raise ValueError(f"Origin node tidak ditemukan di osrm_node_index.csv: {origin_id}")
if dest_node.empty:
    raise ValueError(f"Destination node tidak ditemukan di osrm_node_index.csv: {destination_id}")

origin_node = origin_node.iloc[0]
dest_node = dest_node.iloc[0]

origin_lat, origin_lon = float(origin_node["latitude"]), float(origin_node["longitude"])
dest_lat, dest_lon = float(dest_node["latitude"]), float(dest_node["longitude"])

center_lat = (origin_lat + dest_lat) / 2
center_lon = (origin_lon + dest_lon) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="OpenStreetMap")

origin_popup = f"""
<b>Origin</b><br>
ID: {origin_id}<br>
Type: {origin_node.get('node_type', '')}<br>
Lat/Lon: {origin_lat:.6f}, {origin_lon:.6f}
"""

dest_popup = f"""
<b>Destination</b><br>
ID: {destination_id}<br>
Type: {dest_node.get('node_type', '')}<br>
Distance OSRM: {distance_km:.2f} km<br>
Base duration: {duration_min:.2f} min<br>
p50: {travel_p50_calc:.2f} min<br>
p85: {travel_p85_calc:.2f} min<br>
p95: {travel_p95_calc:.2f} min
"""

folium.Marker(
    [origin_lat, origin_lon],
    popup=folium.Popup(origin_popup, max_width=320),
    tooltip=f"Origin: {origin_id}",
    icon=folium.Icon(icon="home", prefix="fa")
).add_to(m)

folium.Marker(
    [dest_lat, dest_lon],
    popup=folium.Popup(dest_popup, max_width=360),
    tooltip=f"Destination: {destination_id}",
    icon=folium.Icon(icon="flag", prefix="fa")
).add_to(m)

route_added = False
route_note = ""
try:
    route_url = (
        "https://router.project-osrm.org/route/v1/driving/"
        f"{origin_lon},{origin_lat};{dest_lon},{dest_lat}"
        "?overview=full&geometries=geojson"
    )
    resp = requests.get(route_url, timeout=20)
    resp.raise_for_status()
    route_json = resp.json()
    if route_json.get("routes"):
        coords_lonlat = route_json["routes"][0]["geometry"]["coordinates"]
        coords_latlon = [(lat, lon) for lon, lat in coords_lonlat]
        folium.PolyLine(
            coords_latlon,
            weight=5,
            opacity=0.8,
            tooltip="OSRM route geometry for illustrative OD pair"
        ).add_to(m)
        route_added = True
        route_note = "OSRM route geometry berhasil ditampilkan."
except Exception as e:
    route_note = f"OSRM route geometry tidak tersedia; fallback ke garis lurus. Detail: {e}"

if not route_added:
    folium.PolyLine(
        [(origin_lat, origin_lon), (dest_lat, dest_lon)],
        weight=4,
        opacity=0.7,
        dash_array="8,8",
        tooltip="Straight-line OD illustration"
    ).add_to(m)

legend_html = f"""
<div style="position: fixed; bottom: 40px; left: 40px; width: 330px; z-index:9999;
            background-color: white; border:2px solid #555; padding: 10px; font-size: 13px;">
<b>Example OD Travel Time</b><br>
Date: {example_date}<br>
Origin: {origin_id}<br>
Destination: {destination_id}<br>
Distance: {distance_km:.2f} km<br>
OSRM base duration: {duration_min:.2f} min<br>
p50: {travel_p50_calc:.2f} min | p85: {travel_p85_calc:.2f} min | p95: {travel_p95_calc:.2f} min<br>
Penalty p85: {penalty_p85:.2f} min | Penalty p95: {penalty_p95:.2f} min<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

map_path = "/content/example_osrm_quantile_leaflet_map.html"
m.save(map_path)

print(route_note)
print("Saved map:", map_path)
display(m)

## 6. Export Output untuk Lampiran

File yang dihasilkan:
- `example_osrm_quantile_travel_time.csv`
- `example_osrm_quantile_leaflet_map.html`

Kedua file ini dapat dimasukkan ke lampiran artefak digital, misalnya:

```text
Lampiran_C_Saved_Matrix_Package/example_osrm_quantile_travel_time.csv
Lampiran_C_Saved_Matrix_Package/example_osrm_quantile_leaflet_map.html
```

In [ ]:
# ============================================================
# 8. DOWNLOAD OUTPUT FILES
# ============================================================

from google.colab import files

print("Output files:")
print("-", example_csv_path)
print("-", map_path)

# Uncomment jika ingin langsung download.
# files.download(example_csv_path)
# files.download(map_path)

## 7. Teks Interpretasi Siap Masuk Draft RKP

Kalimat ringkas yang dapat digunakan pada Bab 3/Bab 4:

> Untuk memperjelas pembentukan matriks waktu tempuh kuantil, penelitian ini menampilkan contoh komputasional pada satu tanggal pengiriman dan satu pasangan origin-destination. Koordinat origin dan destination digunakan untuk memperoleh estimasi jarak dan durasi dasar dari OSRM driving road-network matrix. Durasi dasar tersebut kemudian dikalikan dengan TomTom-based traffic multiplier p50, p85, dan p95. Dengan pendekatan ini, OSRM berperan sebagai pembentuk estimasi jarak dan durasi dasar berbasis jaringan jalan, sedangkan multiplier berperan sebagai faktor inflasi waktu tempuh untuk merepresentasikan kondisi perjalanan normal, padat, dan sangat padat.

> Pada contoh tersebut, waktu tempuh p50 merepresentasikan kondisi normal atau median, p85 merepresentasikan kondisi padat, dan p95 merepresentasikan kondisi sangat padat atau konservatif tinggi. Ilustrasi ini menunjukkan bahwa perubahan kuantil memiliki dampak langsung terhadap waktu kedatangan, keterlambatan, dan on-time delivery. Oleh karena itu, penggunaan p50, p85, dan p95 tidak dimaknai sebagai prediksi lalu lintas real-time penuh, melainkan sebagai representasi skenario risiko perjalanan berbasis kuantil yang dapat digunakan untuk mengevaluasi reliabilitas rute.